# Lab: Preprocess Data

In [1]:
import re
import unicodedata
from ai_foundations.feedback.course_2 import preprocess

### HTML tags

> **Info: Regular expressions**
>
> A **regular expression**, or "regex," is a special sequence of characters that defines a search pattern. It essentially acts as a powerful and flexible "find and replace" tool for text. For your task of stripping HTML tags from raw text, you can use a regular expression to create a pattern that precisely identifies the structure of any HTML tag, which typically starts with a `<` and ends with a `>`. In Python, you will use the built-in [`re` module](https://docs.python.org/3/howto/regex.html#regex-howto) to apply this pattern. By using a function like `re.sub()`, you can instruct Python to find all substrings in your raw text that match this HTML tag pattern and replace them with an empty string (`""`). This effectively deletes the tags and leaves you with only the clean, human-readable content.
>

In [9]:
# Remove every HTML tag in a string
def clean_html_tags(text: str) -> str:
    regex_pattern = r"<.*?>"
    return re.sub(regex_pattern, "", text)

# Test
text = (
    "<h3>Let's come together to <strong>win</strong> this match."
    " <br>We can do it!</h3>"
)
cleaned_text = clean_html_tags(text)

print(f"Before cleaning: \n\ttext: {text}\n")
print(f"After cleaning: \n\ttext: {cleaned_text}")

Before cleaning: 
	text: <h3>Let's come together to <strong>win</strong> this match. <br>We can do it!</h3>

After cleaning: 
	text: Let's come together to win this match. We can do it!


### Coding Activity 1: Other HTML entities

In [10]:
text = (
    "<p>The Krowor Municipal District was carved out of the Ledzokuku-Krowor"
    " Municipal District in 2018 &amp; it's population is &gt; 200000.</p>"
)
cleaned_text = clean_html_tags(text)
print(cleaned_text)

The Krowor Municipal District was carved out of the Ledzokuku-Krowor Municipal District in 2018 &amp; it's population is &gt; 200000.


We also want to remove &..;

> **Your task:**
>
> Complete the function `clean_html` below which removes HTML tags and other common entities.
>
> Additionally to stripping HTML tags, it should perform the following replacements:
> 1. `"&nbsp;"` should be replaced with `" "`.
> 2. `"&amp;"` should be replaced with `"&"`.
> 3. `"&lt;"` should be replaced with `"<"`.
> 4. `"&gt;"` should be replaced with `">"`.
>

In [15]:
# Strip basic HTML markup and common entities from a string
def clean_html(text: str) -> str:

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Replace HTML entities
    text = re.sub(r"&nbsp;", " ", text)
    text = re.sub(r"&amp;", "&", text)
    text = re.sub(r"&lt;", "<", text)
    text = re.sub(r"&gt;", ">", text)

    return text

In [16]:
# Tests
text = (
    "<p>The Krowor Municipal District was carved out of the Ledzokuku-Krowor"
    " Municipal District in 2018 &amp; it's population is &gt; 200000.</p>"
)

cleaned_text = clean_html(text)

print(f"Before cleaning: \n\ttext: {text}\n")
print(f"After cleaning: \n\ttext: {cleaned_text}\n")

preprocess.test_clean_html(clean_html)

Before cleaning: 
	text: <p>The Krowor Municipal District was carved out of the Ledzokuku-Krowor Municipal District in 2018 &amp; it's population is &gt; 200000.</p>

After cleaning: 
	text: The Krowor Municipal District was carved out of the Ledzokuku-Krowor Municipal District in 2018 & it's population is > 200000.

✅ Nice! Your answer looks correct.


Unicode permet d'attribuer aux symboles, lettres, signes de ponctuation, emojis, ecritures arabes, chinois,... un numero d'identification unique appele `code point`. Donc en realite quand on voit un emoji, ce n'est reellement pas un emoji mais un caractere dont on connait le `code point`.
Exemple: 😱 => `U+1F631`

In [18]:
# Get the unicaode value for any non-text item
hex(ord('😱'))

'0x1f631'

Chaque caractère Unicode (lettres, chiffres, emojis, codes de contrôle, etc.) est identifié par un code composé de deux lettres. La première lettre indique le groupe auquel appartient le caractère, tandis que la deuxième lettre précise sa sous-catégorie. 

In [20]:
print('Symbol 😊\'s Unicode category is:', unicodedata.category('😊'))
print('Symbol 😱\'s Unicode category is:', unicodedata.category('😱'))

Symbol 😊's Unicode category is: So
Symbol 😱's Unicode category is: So


> **Info: `unicodedata` package**
>
> `unicodedata`is a Python module that lets you look up information about any Unicode character. You can use this module to loop up an emoji's formal name, its numeric value (if it has one), and its general category. This category is particularly useful for cleaning texts, as you will see in the next activity.
>


| Category | Meaning            | Common sub-codes & examples |
|-------------|--------------------|-----------------------------|
| **L\***     | Letter             | `Lu` = uppercase (A), `Ll` = lowercase (a), `Lt` = titlecase (ǅ), `Lm` = modifier (ʰ), `Lo` = other letters (汉, ע) |
| **N\***     | Number             | `Nd` = decimal digits (0-9, ٠–٩), `No` = other numbers (½, Ⅻ) |
| **P\***     | Punctuation        | `Po` = other punctuation (!, ?), `Pd` = dash (—), `Ps`/`Pf`/`Pe` = start/final/end brackets |
| **S\***     | Symbol             | `Sm` = math (±, √), `Sc` = currency (₦, $), `Sk` = modifier (ˆ), `So` = other symbols (😊, ⭐) |
| **Z\***     | Separator          | `Zs` = space, `Zl` = line, `Zp` = paragraph |
| **C\***     | Other / Control    | `Cc` = control codes (newline, tab), `Cf` = formatting marks (zero-width joiner), `Cs` = surrogates, `Co`/`Cn` = private-use or unassigned |



In [21]:
# Removes non-text unicode characters from a string
def clean_unicode(text: str) -> str:
    
    # Preserve letters, numbers and punctuation marks.
    categories_to_keep = {"L", "N", "P"}

    keep = []
    for ch in text:
        do_keep = ch.isspace()
        if not do_keep:
            for category in categories_to_keep:
                if unicodedata.category(ch).startswith(category):
                    do_keep = True
                    break
        if do_keep:
            keep.append(ch)
            
    return "".join(keep)


#Test
text = "Bag of rice now cost ₦150000 naira. Ah! 😱 Èdakun o"
cleaned_text = clean_unicode(text)
print(f"Before cleaning: \n\ttext: {text}\n")
print(f"After cleaning: \n\ttext: {cleaned_text}")

preprocess.test_clean_unicode(clean_unicode)

Before cleaning: 
	text: Bag of rice now cost ₦150000 naira. Ah! 😱 Èdakun o

After cleaning: 
	text: Bag of rice now cost 150000 naira. Ah!  Èdakun o
✅ Nice! Your answer looks correct.


How you clean text depends entirely on your project's goal. For instance, when processing social media data, you might want to:

* **Process emojis**: You could strip them out completely, or you could replace them with descriptive tags like `<emoji_happy>` or `<emoji_sad>` for sentiment analysis.

* **Process hashtags**: Instead of removing them, you could extract the text by just dropping the `#` symbol.

* **Map special characters**: You could replace all currency symbols with a special `<money>` tag or punctuation marks such as `<3` with a `<heart>` tag.

Each of these choices customizes the cleaning process for a specific application, whether it's chatbots, analyzing customer feedback, or developing a model that generates poems.

Ultimately, the way you clean your data directly influences what your model learns. Each decision, whether normalizing slang, handling punctuation, or preserving cultural symbols, must be deliberate and aligned with your goal. While automated tools are great for streamlining the process, manual review is essential. This is especially the case for nuanced or culturally rich content. A machine may not be able to infer what information in a text is meaningful for a specific task and that judgment often requires a human touch.


## Summary

In this lab, I gained experience with processing noisy data and learned how to clean HTML and unicode elements. For cleaning HTML, I used **regular expressions**. For removing or replacing unicode elements, I used the `unicodedata` package.

In the next module, we will explore **tokenization**, the process of breaking text into smaller units. Building on my understanding of space tokenizers from the previous course, we will explore more advanced tokenization techniques. These offer greater flexibility, especially when handling previously unseen or rare words.

---